In [1]:
pip install google-cloud-speech yt-dlp pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install imageio-ffmpeg

In [4]:
!~/google-cloud-sdk/bin/gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=CT28JdEQfZtgINYJd6KZ4JXvUrSyHD&access_type=offline&code_challenge=JIPVWnala37iZQmFi4hFLEL0W1i_hzRMKvskOcBLQHY&code_challenge_method=S256


Credentials saved to file: [/Users/a0979/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "project-5125cd39-21bb-43e3-b41" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [8]:
from google.api_core.client_options import ClientOptions

In [5]:
import os
import glob
import pandas as pd
import yt_dlp
import time
import imageio_ffmpeg
import subprocess
import wave
import math
from google.cloud.speech_v2 import SpeechClient
from google.cloud.speech_v2.types import cloud_speech


credentials_path = "/Users/a0979/.config/gcloud/application_default_credentials.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = credentials_path
PROJECT_ID = "project-5125cd39-21bb-43e3-b41" 


def download_audio_from_tiktok(video_url, output_filename="temp_audio.wav"):
    ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
    

    ydl_opts = {
        'format': 'best',
        'outtmpl': 'temp_raw.%(ext)s',
        'quiet': True,
        'no_warnings': True
    }
    
    try:

        for f in glob.glob("temp_raw.*"):
            os.remove(f)
        if os.path.exists(output_filename):
            os.remove(output_filename) 
            

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            error_code = ydl.download([video_url])
        
        if error_code != 0:
            print("Download failed, skip it.")
            return False


        raw_files = glob.glob("temp_raw.*")
        if not raw_files:
            return False
        raw_file = raw_files[0]
        

        command = [
            ffmpeg_path, '-y', '-i', raw_file,
            '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1',
            output_filename
        ]
        
        subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        os.remove(raw_file) 
        
        return True
        
    except Exception as e:
        print(f"Error ({video_url}): {e}")
        return False


def transcribe_audio_v2(audio_file_path="temp_audio.wav"):
    client = SpeechClient()
    full_transcript = []

    try:
        with wave.open(audio_file_path, 'rb') as wf:
            framerate = wf.getframerate()
            nchannels = wf.getnchannels()
            sampwidth = wf.getsampwidth()
            nframes = wf.getnframes()
            

            chunk_length_sec = 30
            frames_per_chunk = int(framerate * chunk_length_sec)
            total_chunks = math.ceil(nframes / float(framerate) / chunk_length_sec)
            
            if total_chunks > 1:
                print(f" Long video, processed by chunks.Total chunks: {total_chunks}")
            
            for i in range(total_chunks):
                chunk_frames = wf.readframes(frames_per_chunk)
                chunk_filename = f"chunk_{i}.wav"
                
                with wave.open(chunk_filename, 'wb') as chunk_wf:
                    chunk_wf.setnchannels(nchannels)
                    chunk_wf.setsampwidth(sampwidth)
                    chunk_wf.setframerate(framerate)
                    chunk_wf.writeframes(chunk_frames)
                    
                with open(chunk_filename, "rb") as f:
                    audio_content = f.read()
                    
                config = cloud_speech.RecognitionConfig(
                    auto_decoding_config=cloud_speech.AutoDetectDecodingConfig(),
                    language_codes=["nl-NL", "fr-FR"], 
                    model="long", 
                )

                request = cloud_speech.RecognizeRequest(
                    recognizer=f"projects/{PROJECT_ID}/locations/global/recognizers/_",
                    config=config,
                    content=audio_content,
                )
                
                try:
                    response = client.recognize(request=request)
                    for result in response.results:
                        full_transcript.append(result.alternatives[0].transcript)
                except Exception as e:
                    print(f" Chunk: {i+1} has error: {e}")
                    
                if os.path.exists(chunk_filename):
                    os.remove(chunk_filename)
                    
        return " ".join(full_transcript)
        
    except Exception as e:
        print(f"STT or chunking error: {e}")
        return ""


def main():
    input_folder = "/Users/a0979/Desktop/tiktok/Data/7052019-09062024/Split_Data/with_account/Scraped_Results"
    search_pattern = os.path.join(input_folder, "*_official.xlsx")
    all_files = glob.glob(search_pattern)

    if not all_files:
        print(f"Can't find the file, please check the path：{input_folder}")
        return
        
    print(f"There are {len(all_files)} files.")

    for file_path in all_files:
        file_name = os.path.basename(file_path)
        if file_name.startswith("~$"): continue
            
        df = pd.read_excel(file_path)
        if "transcription" not in df.columns:
            df["transcription"] = ""

        if df["transcription"].notna().all() and (df["transcription"].astype(str).str.strip() != "").all():
            print(f"File:【{file_name}】is completed, skip it.")
            continue

        print(f"\n" + "="*50)
        print(f"Working on：【{file_name}】")
        print(f"This file contains {len(df)} videos.")

        try:
            for index, row in df.iterrows():
                video_url = row.get('real_video_url', row.get('url')) 
                
                if not video_url or pd.isna(video_url): continue
                

                if pd.notna(row['transcription']) and str(row['transcription']).strip() != "":
                    continue
                    
                print(f"[{index+1}/{len(df)}] Working on: {video_url}")
                

                success = download_audio_from_tiktok(video_url, "temp_audio.wav")
                if not success:
                    df.at[index, 'transcription'] = "DOWNLOAD_FAILED"
                    df.to_excel(file_path, index=False) 
                    continue
                    
     
                print("  Speech downloaded, processing via Google Cloud...")
                transcript = transcribe_audio_v2("temp_audio.wav")
                
                if transcript.strip():
                    print(f"  ✅ Transcription completed: {transcript[:50]}...") 
                    df.at[index, 'transcription'] = transcript
                else:
                    print(" No clear human voice.")
                    df.at[index, 'transcription'] = "NO_SPEECH_DETECTED"
                    
      
                df.to_excel(file_path, index=False)
                time.sleep(2)

        except Exception as e:
            print(f"File {file_name} has error: {e}")

    if os.path.exists("temp_audio.wav"):
        os.remove("temp_audio.wav")
        
    print("\n✅ Completed")

if __name__ == "__main__":
    main()

There are 8 files.
File:【Open VLD_official.xlsx】is completed, skip it.
File:【Groen_official.xlsx】is completed, skip it.
File:【PVDA_official.xlsx】is completed, skip it.
File:【CDandV_official.xlsx】is completed, skip it.

Working on：【N-VA_official.xlsx】
This file contains 61 videos.
[11/61] Working on: https://www.tiktok.com/@annickderidder.official/video/7205637927794134277
  Speech downloaded, processing via Google Cloud...        
 Long video, processed by chunks.Total chunks: 4
  ✅ Transcription completed: in Antwerpen worden een aantal bomen verplaatst vo...
[12/61] Working on: https://www.tiktok.com/@annickderidder.official/video/7058532304406252806
  Speech downloaded, processing via Google Cloud...      
 Long video, processed by chunks.Total chunks: 6
  ✅ Transcription completed: Waar ligt de grootste Sluis van de wereld wel de t...
[13/61] Working on: https://www.tiktok.com/@bjanseeuw/video/7236030637944212763
  Speech downloaded, processing via Google Cloud...      
 No clear h

ERROR: unable to download video data: HTTP Error 404: Not Found


Error (https://www.tiktok.com/@dominieksneppe/video/7335164151192358177): ERROR: unable to download video data: HTTP Error 404: Not Found
[19/245] Working on: https://www.tiktok.com/@dominieksneppe/video/7053416977448176901
  Speech downloaded, processing via Google Cloud...        
 Long video, processed by chunks.Total chunks: 2
  ✅ Transcription completed: en in een mijn tijd kan dus wel de epidemiologisch...
[20/245] Working on: https://www.tiktok.com/@adelineblancquaer/video/7366686284762418464
  Speech downloaded, processing via Google Cloud...      
 Long video, processed by chunks.Total chunks: 2
  ✅ Transcription completed: Renault clip van het is sector die is wel zeer dui...
[21/245] Working on: https://www.tiktok.com/@adelineblancquaer/video/7339941830873648417
  Speech downloaded, processing via Google Cloud...      
 Long video, processed by chunks.Total chunks: 2
  ✅ Transcription completed: het vlaamse woonbeleid heeft gefaald meer dan 170....
[22/245] Working on: https

ERROR: unable to download video data: HTTP Error 404: Not Found


Error (https://www.tiktok.com/@reccino/video/7339506967834070304): ERROR: unable to download video data: HTTP Error 404: Not Found
[65/245] Working on: https://www.tiktok.com/@reccino/video/7329256292269313312
  Speech downloaded, processing via Google Cloud...      
  ✅ Transcription completed: deze week werd het stikstof ik kreeg goedgekeurd i...
[66/245] Working on: https://www.tiktok.com/@reccino/video/7325566234748128544
  Speech downloaded, processing via Google Cloud...        
 Long video, processed by chunks.Total chunks: 2
  ✅ Transcription completed: het Vlaams belang komt gelukkig wel op voor die ge...
[67/245] Working on: https://www.tiktok.com/@reccino/video/7309841824229625120
  Speech downloaded, processing via Google Cloud...      
 Long video, processed by chunks.Total chunks: 2
  ✅ Transcription completed: het Vlaams belang voert hier vandaag in de Gasthui...
[68/245] Working on: https://www.tiktok.com/@reccino/video/7305378908977761569


...

Error (https://www.tiktok.com/@pieter.de.spiegeleer/video/7317339302311087393): ERROR: [TikTok] 7317339302311087393: No video formats found!; please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U
[76/245] Working on: https://www.tiktok.com/@pieter.de.spiegeleer/video/7314814450471636257


ERROR: [TikTok] 7314814450471636257: No video formats found!; please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U


Error (https://www.tiktok.com/@pieter.de.spiegeleer/video/7314814450471636257): ERROR: [TikTok] 7314814450471636257: No video formats found!; please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U
[77/245] Working on: https://www.tiktok.com/@johandeckmyn/video/7375127219712118048
  Speech downloaded, processing via Google Cloud...      
 Long video, processed by chunks.Total chunks: 3
  ✅ Transcription completed: ja kijk zitten allemaal feestjes resultaat staat n...
[78/245] Working on: https://www.tiktok.com/@johandeckmyn/video/7369193043909561633
  Speech downloaded, processing via Google Cloud...      
 Long video, processed by chunks.Total chunks: 2
  ✅ Transcription completed: ik ben Johan dat ik mijn en de komende verkiezinge...
[79/245] Working on: https://www.tiktok.com/@johandeckmyn/video/7364709318052121888
Error (https://www.tiktok.com/@johandeckmyn/video/736

ERROR: unable to download video data: HTTP Error 404: Not Found


Error (https://www.tiktok.com/@filipbrusselmans/video/7307324345020452128): ERROR: unable to download video data: HTTP Error 404: Not Found
[171/245] Working on: https://www.tiktok.com/@filipbrusselmans/video/7293112160819973408
  Speech downloaded, processing via Google Cloud...        
 No clear human voice.
[172/245] Working on: https://www.tiktok.com/@filipbrusselmans/video/7288789814319648033
  Speech downloaded, processing via Google Cloud...      
 Long video, processed by chunks.Total chunks: 3
  ✅ Transcription completed: Ik had enkele dagen geleden eergisteren mevrouw pe...
[173/245] Working on: https://www.tiktok.com/@filipbrusselmans/video/7286858513924541729
  Speech downloaded, processing via Google Cloud...      
 Long video, processed by chunks.Total chunks: 4
  ✅ Transcription completed: of het nu Franken de blak Medi of Nicole is het mo...
[174/245] Working on: https://www.tiktok.com/@filipbrusselmans/video/7286580987025296673
  Speech downloaded, processing via Googl

In [1]:
import os
import glob
import pandas as pd
import yt_dlp
import time
import imageio_ffmpeg
import subprocess
import wave
import math
from google.cloud.speech_v2 import SpeechClient
from google.cloud.speech_v2.types import cloud_speech


credentials_path = "/Users/a0979/.config/gcloud/application_default_credentials.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = credentials_path
PROJECT_ID = "project-5125cd39-21bb-43e3-b41" 


def download_audio_from_tiktok(video_url, output_filename="temp_audio.wav"):
    ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
    

    ydl_opts = {
        'format': 'best',
        'outtmpl': 'temp_raw.%(ext)s',
        'quiet': True,
        'no_warnings': True
    }
    
    try:

        for f in glob.glob("temp_raw.*"):
            os.remove(f)
        if os.path.exists(output_filename):
            os.remove(output_filename) 
            

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            error_code = ydl.download([video_url])
        
        if error_code != 0:
            print("Download failed, skip it.")
            return False


        raw_files = glob.glob("temp_raw.*")
        if not raw_files:
            return False
        raw_file = raw_files[0]
        

        command = [
            ffmpeg_path, '-y', '-i', raw_file,
            '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1',
            output_filename
        ]
        
        subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        os.remove(raw_file) 
        
        return True
        
    except Exception as e:
        print(f"Error ({video_url}): {e}")
        return False


def transcribe_audio_v2(audio_file_path="temp_audio.wav"):
    client = SpeechClient()
    full_transcript = []

    try:
        with wave.open(audio_file_path, 'rb') as wf:
            framerate = wf.getframerate()
            nchannels = wf.getnchannels()
            sampwidth = wf.getsampwidth()
            nframes = wf.getnframes()
            

            chunk_length_sec = 30
            frames_per_chunk = int(framerate * chunk_length_sec)
            total_chunks = math.ceil(nframes / float(framerate) / chunk_length_sec)
            
            if total_chunks > 1:
                print(f" Long video, processed by chunks.Total chunks: {total_chunks}")
            
            for i in range(total_chunks):
                chunk_frames = wf.readframes(frames_per_chunk)
                chunk_filename = f"chunk_{i}.wav"
                
                with wave.open(chunk_filename, 'wb') as chunk_wf:
                    chunk_wf.setnchannels(nchannels)
                    chunk_wf.setsampwidth(sampwidth)
                    chunk_wf.setframerate(framerate)
                    chunk_wf.writeframes(chunk_frames)
                    
                with open(chunk_filename, "rb") as f:
                    audio_content = f.read()
                    
                config = cloud_speech.RecognitionConfig(
                    auto_decoding_config=cloud_speech.AutoDetectDecodingConfig(),
                    language_codes=["nl-NL", "fr-FR"], 
                    model="long", 
                )

                request = cloud_speech.RecognizeRequest(
                    recognizer=f"projects/{PROJECT_ID}/locations/global/recognizers/_",
                    config=config,
                    content=audio_content,
                )
                
                try:
                    response = client.recognize(request=request)
                    for result in response.results:
                        full_transcript.append(result.alternatives[0].transcript)
                except Exception as e:
                    print(f" Chunk: {i+1} has error: {e}")
                    
                if os.path.exists(chunk_filename):
                    os.remove(chunk_filename)
                    
        return " ".join(full_transcript)
        
    except Exception as e:
        print(f"STT or chunking error: {e}")
        return ""


def main():
    input_folder = "/Users/a0979/Desktop/tiktok/Data/10062024 to present/split data/with_account/Scraped_Results"
    search_pattern = os.path.join(input_folder, "*_official.xlsx")
    all_files = glob.glob(search_pattern)

    if not all_files:
        print(f"Can't find the file, please check the path：{input_folder}")
        return
        
    print(f"There are {len(all_files)} files.")

    for file_path in all_files:
        file_name = os.path.basename(file_path)
        if file_name.startswith("~$"): continue
            
        df = pd.read_excel(file_path)
        if "transcription" not in df.columns:
            df["transcription"] = ""

        if df["transcription"].notna().all() and (df["transcription"].astype(str).str.strip() != "").all():
            print(f"File:【{file_name}】is completed, skip it.")
            continue

        print(f"\n" + "="*50)
        print(f"Working on：【{file_name}】")
        print(f"This file contains {len(df)} videos.")

        try:
            for index, row in df.iterrows():
                video_url = row.get('real_video_url', row.get('url')) 
                
                if not video_url or pd.isna(video_url): continue
                

                if pd.notna(row['transcription']) and str(row['transcription']).strip() != "":
                    continue
                    
                print(f"[{index+1}/{len(df)}] Working on: {video_url}")
                

                success = download_audio_from_tiktok(video_url, "temp_audio.wav")
                if not success:
                    df.at[index, 'transcription'] = "DOWNLOAD_FAILED"
                    df.to_excel(file_path, index=False) 
                    continue
                    
     
                print("  Speech downloaded, processing via Google Cloud...")
                transcript = transcribe_audio_v2("temp_audio.wav")
                
                if transcript.strip():
                    print(f"  ✅ Transcription completed: {transcript[:50]}...") 
                    df.at[index, 'transcription'] = transcript
                else:
                    print(" No clear human voice.")
                    df.at[index, 'transcription'] = "NO_SPEECH_DETECTED"
                    
      
                df.to_excel(file_path, index=False)
                time.sleep(2)

        except Exception as e:
            print(f"File {file_name} has error: {e}")

    if os.path.exists("temp_audio.wav"):
        os.remove("temp_audio.wav")
        
    print("\n✅ Completed")

if __name__ == "__main__":
    main()

There are 8 files.
File:【Team Fouad Ahidar_official.xlsx】is completed, skip it.
File:【Open Vld_official.xlsx】is completed, skip it.
File:【Groen_official.xlsx】is completed, skip it.
File:【PVDA_official.xlsx】is completed, skip it.
File:【cdandv_official.xlsx】is completed, skip it.
File:【N-VA_official.xlsx】is completed, skip it.
File:【Vooruit_official.xlsx】is completed, skip it.

Working on：【Vlaams Belang_official.xlsx】
This file contains 591 videos.
[138/591] Working on: https://www.tiktok.com/@ellen_samyn/video/7634269611659218209
  Speech downloaded, processing via Google Cloud...      
 Long video, processed by chunks.Total chunks: 3
  ✅ Transcription completed: de vraag is echter niet alleen of mensen langer zu...
[139/591] Working on: https://www.tiktok.com/@ellen_samyn/video/7632280096769838369
  Speech downloaded, processing via Google Cloud...      
 Long video, processed by chunks.Total chunks: 2
  ✅ Transcription completed: Vandaag herdenken we de Armeense genocide een eh t...
[

ERROR: [TikTok] 7643793981041544480: No video formats found!; please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U


Error (https://www.tiktok.com/@orredepo/video/7643793981041544480): ERROR: [TikTok] 7643793981041544480: No video formats found!; please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U
[511/591] Working on: https://www.tiktok.com/@orredepo/video/7642028047788543265
  Speech downloaded, processing via Google Cloud...        
 Long video, processed by chunks.Total chunks: 3
  ✅ Transcription completed: Zit u ergens een eh heel en het en het feit dat me...
[512/591] Working on: https://www.tiktok.com/@orredepo/video/7640460660018613537
  Speech downloaded, processing via Google Cloud...      
 Long video, processed by chunks.Total chunks: 3
  ✅ Transcription completed: Ja wij zijn voor vrijwillige fusies van politiezon...
[513/591] Working on: https://www.tiktok.com/@orredepo/video/7639786032313011488
  Speech downloaded, processing via Google Cloud...      
 Long video, p

ERROR: [TikTok] 7630883498139716896: No video formats found!; please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U


Error (https://www.tiktok.com/@orredepo/video/7630883498139716896): ERROR: [TikTok] 7630883498139716896: No video formats found!; please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U
[517/591] Working on: https://www.tiktok.com/@orredepo/video/7630011544335682848


ERROR: [TikTok] 7630011544335682848: No video formats found!; please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U


Error (https://www.tiktok.com/@orredepo/video/7630011544335682848): ERROR: [TikTok] 7630011544335682848: No video formats found!; please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U
[518/591] Working on: https://www.tiktok.com/@orredepo/video/7629988081277422880
  Speech downloaded, processing via Google Cloud...      
 Long video, processed by chunks.Total chunks: 2
  ✅ Transcription completed: wist u dat u de helft van de prijs die u aan de po...
[519/591] Working on: https://www.tiktok.com/@orredepo/video/7629018003501305120
  Speech downloaded, processing via Google Cloud...        
 Long video, processed by chunks.Total chunks: 3
  ✅ Transcription completed: in verband met een eh betoging op donderdag 26 maa...
[520/591] Working on: https://www.tiktok.com/@orredepo/video/7628261425642835233
  Speech downloaded, processing via Google Cloud...      
 Long video, p

ERROR: [TikTok] 7563680886055505184: No video formats found!; please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U


Error (https://www.tiktok.com/@guydhaeseleer1/video/7563680886055505184): ERROR: [TikTok] 7563680886055505184: No video formats found!; please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U
[576/591] Working on: https://www.tiktok.com/@guydhaeseleer1/video/7561697753307319584
  Speech downloaded, processing via Google Cloud...        
 No clear human voice.
[577/591] Working on: https://www.tiktok.com/@guydhaeseleer1/video/7560775585660685601
  Speech downloaded, processing via Google Cloud...        
 No clear human voice.
[578/591] Working on: https://www.tiktok.com/@guydhaeseleer1/video/7560771088649702689
  Speech downloaded, processing via Google Cloud...        
 No clear human voice.
[579/591] Working on: https://www.tiktok.com/@guydhaeseleer1/video/7560298460541455648
  Speech downloaded, processing via Google Cloud...        
 No clear human voice.
[580/591] W